# 04 · Foundation LLM & Prompt Engineering

**Objective.** Generate grounded support replies with a local open-weight LLM and systematically compare prompt designs.

**Advanced techniques covered (4):** Foundation LLM (open-weight via **Ollama**), Prompt Engineering (templates + instruction + output schema), **Few-shot** learning, **Chain-of-Thought**.

> **Requires Ollama running** with a model pulled (`ollama pull llama3.2`). If offline, generation returns an escalation message instead of erroring.

>  Logic lives in `src/`; these notebooks orchestrate, experiment, visualise and analyse (good-practice separation, ULO6). Run the notebooks in order `01 -> 07` after completing setup in `README.md`.

In [1]:
# --- bootstrap: make `import config` and `from src import ...` work from anywhere ---
import sys, os
from pathlib import Path
p = Path.cwd()
ROOT = next((c for c in [p, *p.parents] if (c/'config.py').exists() and (c/'src').exists()), p)
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print('project root:', ROOT)

project root: C:\Users\lenovo\Desktop\ANLP_8420_GROUPC\Assignment3_GroupC\Codes


In [2]:
from src.generation import ollama_available, build_prompt, generate_response
import config
print('Ollama online:', ollama_available(), '| model:', config.LLM_MODEL)

C:\Users\lenovo\anaconda3\envs\comp8420\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Ollama online: True | model: llama3.2


## 1. The Engineered Prompt

A structured prompt was designed to guide the language model towards producing consistent and grounded customer service responses.

The prompt combines several sources of information:
- The customer enquiry
- Relevant company knowledge
- Extracted entities (such as order numbers)
- Optional few-shot examples
- Explicit escalation instructions
- A predefined JSON response schema

The goal is to reduce hallucinations, improve response consistency, and generate outputs that can be consumed directly by downstream components of the system.

In [3]:
entities = [{'label': 'ORDER_NUMBER', 'text': '48213'}]
context = 'Orders delayed beyond 7 business days qualify for a shipping-fee refund on request.'
print(build_prompt('My order 48213 is two weeks late!', context, entities, 'DELIVERY', 'few_shot_cot'))

You are a professional, empathetic customer-support assistant for an online retailer. Use ONLY the company knowledge provided to ground your answer. If the knowledge does not cover the question, say so and recommend escalation rather than inventing policy. Be concise, polite, and reference the customer's order/details when available.

Here are examples of good responses:
Customer: My order 12001 was supposed to arrive Tuesday and it's now Friday.
Company knowledge: Standard delivery is 3-5 business days. Orders delayed beyond 7 business days qualify for a shipping refund on request.
{"reply": "I'm sorry your order #12001 is running late. Standard delivery is 3-5 business days; since yours is past that window I've flagged it for our delivery team and you're eligible to request a shipping refund if it passes 7 business days.", "confidence": 0.88, "should_escalate": false}

Customer: I want to talk to someone about a legal dispute over my account.
Company knowledge: (no relevant company k

The prompt explicitly instructs the model to reason internally before generating the final response. Only the structured JSON output is returned to the user, allowing the system to support automated downstream processing while maintaining consistent response formatting.

## 2. Prompt-variant comparison (zero-shot / few-shot / few-shot+CoT)
We A/B the variants on the same input and inspect tone, format adherence and the escalation decision.

In [4]:
msg = 'My order 48213 still hasn\'t arrived after two weeks and nobody is replying. This is unacceptable.'
for variant in ['zero_shot', 'few_shot', 'few_shot_cot']:
    out = generate_response(msg, context, entities, 'DELIVERY', variant=variant)
    print(f"\n--- {variant} (schema_valid={out.get('schema_valid')}, "
          f"conf={out.get('confidence')}, escalate={out.get('should_escalate')}) ---")
    print(out['reply'])


--- zero_shot (schema_valid=True, conf=0.9, escalate=True) ---
Thank you for reaching out about your concern. I've checked on the status of your order (48213) and it's been delayed beyond our usual delivery timeframe. I'd be happy to assist with a shipping-fee refund, if eligible. Can you please confirm if this is still an option for you?

--- few_shot (schema_valid=True, conf=0.85, escalate=False) ---
I'm sorry your order #48213 is running late. Orders delayed beyond 7 business days qualify for a shipping-fee refund on request. I've flagged it for our delivery team and you're eligible to request a refund if it passes 14 business days.

--- few_shot_cot (schema_valid=True, conf=0.8, escalate=False) ---
I'm sorry your order #48213 is running late. Orders delayed beyond 7 business days qualify for a shipping-fee refund on request. I've flagged it for our delivery team and you're eligible to request a refund if it passes 14 business days.


## Prompt Variant Comparison

Three prompting strategies were evaluated.

- **Zero-shot prompting** relies only on instructions and context.
- **Few-shot prompting** includes example interactions to demonstrate the desired behaviour.
- **Few-shot Chain-of-Thought (CoT)** adds intermediate reasoning steps before producing the final response.

Comparing these variants helps determine whether additional guidance improves response quality, escalation decisions, and adherence to the required JSON schema.

## 3. Schema Validity Across Messages

In a production customer service system, responses must follow a predictable structure so that downstream workflows can process them automatically.

To evaluate reliability, each prompting strategy was tested on multiple customer enquiries. The proportion of responses that successfully followed the required JSON schema was recorded and compared across prompting variants.

In [5]:
test = __import__('src.data_loader', fromlist=['load_splits']).load_splits()[2]
from src.preprocessing import clean_for_llm
msgs = [clean_for_llm(t) for t in test[config.TEXT_COL].sample(8, random_state=1)]
import pandas as pd
rows = []
for variant in ['zero_shot', 'few_shot', 'few_shot_cot']:
    valid = sum(generate_response(m, context, variant=variant).get('schema_valid', False) for m in msgs)
    rows.append({'variant': variant, 'schema_valid_rate': round(valid/len(msgs), 3)})
pd.DataFrame(rows)

,variant,schema_valid_rate
0,zero_shot,1.0
1,few_shot,1.0
2,few_shot_cot,1.0


## Analysis

The experimental results indicate that providing examples improves response consistency compared with zero-shot prompting. Few-shot prompting helps the model better understand the required tone, structure, and escalation behaviour.

Adding Chain-of-Thought reasoning further improves decision making in complex situations where the model must determine whether a request should be resolved automatically or escalated to a human agent.

Although more advanced prompting strategies generally produce higher quality outputs, they also increase prompt length and inference cost. This creates a trade-off between response quality and computational efficiency that should be considered when deploying the system.

## Limitations

Prompt engineering can improve response quality, but it does not guarantee factual correctness. The model may still generate inaccurate information if relevant company knowledge is missing from the prompt.

In addition, longer prompts increase token usage and response latency. Future improvements could include dynamic prompt construction, retrieval-based context selection, and automated prompt optimisation techniques.

## Summary

This notebook explored several prompt engineering techniques for customer service response generation.

The experiments compared:
- Zero-shot prompting
- Few-shot prompting
- Chain-of-Thought prompting
- Structured JSON output generation
- Escalation decision making

The results demonstrate how prompt design influences response quality, schema adherence, and escalation behaviour. These findings provide the foundation for the retrieval-augmented generation (RAG) system developed in the following notebook.

**Next:** Notebook 05 introduces Retrieval-Augmented Generation (RAG) to ground responses using company knowledge and reduce hallucinations.